[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/hyperspectral-course-2026/blob/main/exercises/ex_00_data_exploration.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🛰️ Exercises: Tanager data exploration
### Space Academy 2026 · Hyperspectral course · Week 1 — Hyperspectral data access, visualization, and exploration (Practical)

Practise the basic moves of hyperspectral data handling on the coastal El Salvador scene from the quick-start notebook: choosing bands, reading spectra at map coordinates, judging how sensitive an index is to band choice, sampling large cubes efficiently, and masking before interpreting.

These exercises accompany the theme notebook [`00_EO_data_quickstart_notebook.ipynb`](../../notebooks/00_EO_data_quickstart_notebook.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Matching target wavelengths to real Tanager bands (centre and FWHM)
- Converting longitude/latitude to pixel positions and plotting pixel spectra
- Testing how sensitive an index is to the exact bands used
- Using strided reads for fast, representative statistics
- Masking before interpreting (water-only NDCI)

**Data**
- `20250223_165546_32_4001` — Usulután coast, El Salvador (mangroves, estuary, ocean, cropland)

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
SCENE_ID = "20250223_165546_32_4001"   # Usulután coast, El Salvador (coastal-water-bodies)
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
# The four bands used for the indices in notebook 00
R560, R665, R708, R800 = (scene.read_band(w) for w in (560, 665, 708, 800))
R860 = scene.read_band(860)
NDWI = hu.nd(R560, R860)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(scene.rgb())
ax.set_title("True colour (640 / 550 / 460 nm)"); ax.axis("off")
plt.show()

## Exercise 1 — Which band is 'the red-edge band'?  ⭐

Notebook 00 asks for bands at 560, 665, 708 and 800 nm, but Tanager has no band *exactly* at those
wavelengths — it takes the closest one.

1. For each target wavelength (443, 560, 665, 708, 800, 1650 nm), print the band index, the actual centre
   wavelength, the difference from the target and the band's FWHM (full width at half maximum).
2. Sentinel-2 band B4 (red) is centred at 664.6 nm and is about 31 nm wide. How many Tanager bands fall
   inside B4 (centre ± FWHM/2)?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`hu.band_index(scene.wavelengths, target)` gives the index; `scene.wavelengths[i]` and `scene.fwhm[i]`
give the centre and width. For part 2 use a boolean mask on `scene.wavelengths`.

</details>

## Exercise 2 — A pixel spectrum from a map coordinate  ⭐

Field teams and partners give you **longitude/latitude**, not row/column. Plot the spectra of these three
points on one figure:

| Point | Longitude | Latitude |
|---|---|---|
| Mangrove forest | −88.4404 | 13.2276 |
| Turbid estuary water | −88.4427 | 13.1631 |
| Sugar-cane field | −88.5104 | 13.2739 |

Use `hu.plot_spectrum` so the lines break at the removed water-vapour bands. Then describe in one sentence
per spectrum *why* it has the shape it has.

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`scene.lonlat_to_rowcol(lon, lat)` returns arrays of rows and columns; `scene.read_pixel(row, col)` returns
`(spectrum, wavelengths)` with the bad bands already removed. `hu.shade_bad_regions(ax)` greys out the
removed windows.

</details>

## Exercise 3 — How much does the band choice change NDVI?  ⭐⭐

Notebook 00 computes NDVI with NIR = 800 nm and red = 665 nm. Other notebooks use NIR = 860 nm.
Compute both versions over the scene and quantify the difference:

1. Mean and 95th percentile of |NDVI₈₀₀ − NDVI₈₆₀| over valid pixels.
2. The same statistic separately for vegetation (NDVI₈₀₀ > 0.5) and water (NDWI > 0.1).
3. A scatter plot (or 2-D histogram) of one version against the other.

When does the band choice matter, and when doesn't it?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Read the 860 nm band with `scene.read_band(860)`. `plt.hist2d(x, y, bins=200, cmin=1)` handles
hundreds of thousands of points well. Remember to drop NaNs: `ok = np.isfinite(a) & np.isfinite(b)`.

</details>

## Exercise 4 — Strided reads: fast previews that are good enough  ⭐⭐

Notebook 00 computes the mean spectrum with `stride = 8` (every 8th row and column, i.e. 1/64 of the pixels).
Is that accurate enough?

1. Compute the scene-mean spectrum (valid pixels, good bands) with strides 16, 8 and 4, and time each read.
2. Use stride 2 as the reference and report, for each stride, the root-mean-square difference to the reference
   in reflectance units and in % of the mean reflectance.

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`cube, wl = scene.read_cube(stride=s)` already masks invalid pixels with NaN, so
`np.nanmean(cube, axis=(1, 2))` is the mean spectrum. Time with `time.perf_counter()`.

</details>

## Exercise 5 — Chlorophyll (NDCI) over water only  ⭐⭐⭐

NDCI = (R₇₀₈ − R₆₆₅) / (R₇₀₈ + R₆₆₅) is a **water** index (Mishra & Mishra, 2012). Over land it simply lights
up vegetation, which makes the colour scale in notebook 00 useless for water.

1. Build a water mask (NDWI > 0.1 and valid) and compute NDCI only for water pixels.
2. Map it with a colour scale chosen from the water pixels' own 2nd–98th percentiles.
3. Compare the NDCI distribution of the **estuary / lagoon** with the **open ocean**. The open ocean lies
   south of the barrier beach, which runs roughly along the pixel line `row = 460 + 0.21 × col`
   (water pixels below that line = ocean).
4. Is the higher NDCI in one of them necessarily more chlorophyll? What else raises R₇₀₈ relative to R₆₆₅?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`rows, cols = np.indices(water.shape)` gives the row and column of every pixel, so
`ocean = water & (rows > 460 + 0.21 * cols)`. For histograms use
`plt.hist(values, bins=100, histtype="step", density=True)`.

</details>

## Exercise 6 — Reflection — what is 'the scene mean' actually averaging?  ⭐

Answer in a few sentences (markdown cell):

1. What fraction of the grid is usable (valid) and why is it not 100% even though the cloud cover is 0%?
2. The theme notebook plots **one mean spectrum for the whole scene**. Roughly what share of the valid
   pixels is water, and how does that shape the mean spectrum? When is a scene-mean spectrum misleading?

In [ ]:
# compute the numbers you need here, then answer in a markdown cell below

<details><summary>💡 Hint</summary>

Compare `scene.masks()["nodata"].mean()` with `valid.mean()`, and compute `(valid & (NDWI > 0.1)).sum() / valid.sum()`.

</details>

## Key takeaways

- Tanager's ~5 nm bands sample narrow features that a multispectral band averages away.
- Always state the exact wavelengths of an index — thresholds are not transferable otherwise.
- Strided reads give representative statistics quickly, but not spatial detail.
- Mask first (water, vegetation, valid pixels), then interpret; a scene mean mixes everything.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC